In [1]:
#1)VectorStoreRetriever	Vector-based	Embedding similarity search	General-purpose RAG
# ConversationalRetrievalChain
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langgraph.checkpoint.memory import InMemorySaver
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
import os
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_openai import ChatOpenAI
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.agents import create_react_agent, AgentExecutor, initialize_agent, AgentType

# 1. Setup
import os
from dotenv import load_dotenv

# 2.Load API keys
load_dotenv(".env")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


# 2. Vector DB
with open("sample.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

# 🧠 Split the text into smaller chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
texts = splitter.split_text(text_data)

embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_texts(texts, embedding)
retriever = vectorstore.as_retriever()

# 3. Conversational RAG chain
rag_chain = ConversationalRetrievalChain.from_llm(
    # Fixed model name typo from gpt-4.1-mini to gpt-4o-mini
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0), 
    retriever=retriever,
    return_source_documents=False
)
# 4. Memory for chat history
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# 5. Wrap RAG as a StructuredTool
def rag_tool_fn(question: str) -> str:
    return rag_chain.invoke({
        "question": question,
        "chat_history": []
    })["answer"]

from langchain_core.tools import StructuredTool
rag_tool = StructuredTool.from_function(
    name="RAG_QA",
    description="Use this to answer questions about LangChain.",
    func=rag_tool_fn
)

# 6. Create agent
agent = initialize_agent(
    tools=[rag_tool],
    llm=ChatOpenAI(model="gpt-4.1-mini", temperature=0),
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True,
    memory=memory,
    handle_parsing_errors=True
)

# 7. Run conversation
print("1️⃣ First question")
res1 = agent.run("What is LangChain?")
print("Answer:", res1)

print("\n2️⃣ Follow-up")
res2 = agent.run("Who created it?")
print("Answer:", res2)

print("\n3️⃣ Ask again")
res3 = agent.run("Explain LangChain again simply.")
print("Answer:", res3)

/var/folders/pr/q76x9dw10ds6vb4_8y13__tr0000gn/T/ipykernel_46136/2346806512.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
/var/folders/pr/q76x9dw10ds6vb4_8y13__tr0000gn/T/ipykernel_46136/2346806512.py:35: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/var/folders/pr/q76x9dw10ds6vb4_8y13__tr0000gn/T/ipykernel_46136/2346806512.py:47: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
/var/folders/pr/q76x9dw10ds6vb4_8y13__tr0000gn/T/ipykernel_46136/2346806512.py:64: LangChainDeprecationWarning: Use `langchain.agents.create_agent` for new applications. It provides a more flexible agent factory with middleware support, structured output, and integration with LangGraph for persistence, streaming, and human-in-the-loop workflows. Migration guide: https://docs.langchain.com/oss/python/migr

1️⃣ First question


> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes  
Action: RAG_QA  
Action Input: What is LangChain?  
Observation: LangChain is a framework for building applications with large language models (LLMs). It was created by Harrison Chase and supports various features such as retrieval-augmented generation (RAG), agents, memory, tools, and more. LangChain is commonly used in applications like chatbots, document question and answer systems, and AI workflows.
Thought:Thought: Do I need to use a tool? No  
AI: LangChain is a framework designed for building applications that use large language models (LLMs). Created by Harrison Chase, it supports features like retrieval-augmented generation (RAG), agents, memory, and integration with various tools. LangChain is commonly used to develop chatbots, document question-answering systems, and AI-driven workflows, making it easier to build complex applications that leverage LLMs effectively.

> Finished 